In [1]:
# Instale as dependências deste notebook no kernel atual, se necessário.
%pip install requests pandas


Note: you may need to restart the kernel to use updated packages.


# Exploração inicial — API do Senado

O Senado mantém catálogo de conjuntos e uma API legislativa com Swagger/OpenAPI. Os formatos e caminhos podem variar por serviço. Este notebook consulta a especificação e duas rotas de amostra; mantenha os volumes baixos. Consulte `docs/fontes_e_endpoints.md`.

In [2]:
import requests
import pandas as pd
from pprint import pprint

BASE = 'https://legis.senado.leg.br/dadosabertos'
session = requests.Session()
session.headers.update({'Accept': 'application/json', 'User-Agent': 'exploracao-painel-legislativo/0.1'})

def get_json(url_or_path, params=None):
    url = url_or_path if url_or_path.startswith('http') else f'{BASE}/{url_or_path.lstrip("/")}'
    response = session.get(url, params=params, timeout=45)
    print(response.status_code, response.url)
    response.raise_for_status()
    return response.json()


## 1. Especificação OpenAPI e inventário

A rota do Swagger UI aponta para OpenAPI 3.1 em `/v3/api-docs`. Extraia famílias e parâmetros exatos da especificação corrente.

In [3]:
spec = get_json(f'{BASE}/v3/api-docs')
print(spec.get('openapi'), spec.get('info'))
servers = spec.get('servers', [])
print('servers:', servers)
rows = []
for path, operations in spec.get('paths', {}).items():
    for method, operation in operations.items():
        if method.lower() in {'get', 'post', 'put', 'delete'}:
            rows.append({'tag': ', '.join(operation.get('tags', [])), 'method': method.upper(), 'path': path, 'summary': operation.get('summary', '')})
routes_df = pd.DataFrame(rows).sort_values(['tag', 'path']).reset_index(drop=True)
display(routes_df)


200 https://legis.senado.leg.br/dadosabertos/v3/api-docs
3.1.0 {'title': 'Dados Abertos Legislativos do Senado Federal e Congresso Nacional', 'description': ' <p>A API de Dados Abertos Legislativos do Senado Federal e do Congresso Nacional é uma API\n de acesso público,\n sem necessidade de autenticação. Contempla informações sobre Parlamentares, reuniões de\n Comissão e Sessões Plenárias, Processos Legislativos, Votações, Discursos, Orçamento\n e diversas outras informações relacionadas ao Processo Legislativo no âmbito do Congresso Nacional Brasileiro.</p>\n <p> Para outras APIs de Dados Abertos do Senado Federal, acesse\n <a href="https://www12.senado.leg.br/dados-abertos">esta página</a>.</p>\n\n**Como selecionar formatos de saída**\n\nDependendo do formato de saída disponível no campo "Media type" abaixo,\nefetue a requisição acrescentando o cabeçalho HTTP *Accept* com um dos formatos disponível\npara o serviço no campo "Media type".\n\nPor exemplo: Se existir o formato "applicati

,tag,method,path,summary
0,Comissão,GET,/dadosabertos/comissao/agenda/atual/iCal,Obtém a Agenda das Comissões do Congresso Naci...
1,Comissão,GET,/dadosabertos/comissao/agenda/mes/{mesReferencia},Agenda das Reuniões de Comissão no mês informado
2,Comissão,GET,/dadosabertos/comissao/agenda/{dataInicio}/{da...,Agenda das Reuniões de Comissão
3,Comissão,GET,/dadosabertos/comissao/agenda/{dataReferencia},Agenda das Reuniões de Comissão
4,Comissão,GET,/dadosabertos/comissao/cpi/{comissao}/requerim...,Lista Requerimentos de CPI. Exclusivo para CPIs
...,...,...,...,...
152,Votação,GET,/dadosabertos/senador/{codigo}/votacoes,(DEPRECATED) Obtém as Votações Nominais de um ...
153,Votação,GET,/dadosabertos/votacao,Votos Nominais de Parlamentares em Processos L...
154,Votação,GET,/dadosabertos/votacaoComissao/comissao/{siglaC...,Votações de Matérias nas Comissões do Senado F...
155,Votação,GET,/dadosabertos/votacaoComissao/materia/{sigla}/...,Votações de Matérias nas Comissões por Identif...


## 2. Testar lista de senadores

Este endpoint de lista respondeu durante o levantamento. Observe o envelope completo (não assuma o mesmo formato da Câmara) e avalie se o payload é JSON estrutural ou XML serializado dentro do JSON.

In [4]:
senators = get_json('senador/lista/atual.json')
print('tipo:', type(senators))
pprint(senators)
# Normalização exploratória; estruturas XML-JSON podem exigir parsing próprio.
display(pd.json_normalize(senators, sep='.').T.head(100))


200 https://legis.senado.leg.br/dadosabertos/senador/lista/atual.json
tipo: <class 'dict'>
{'ListaParlamentarEmExercicio': {'Metadados': {'DataVersaoServico': '2020-07-15',
                                               'DescricaoDataSet': 'Lista dos '
                                                                   'Parlamentares '
                                                                   'que estão '
                                                                   'atualmente '
                                                                   'em '
                                                                   'Exercício. '
                                                                   'Informações '
                                                                   'atuais '
                                                                   'sobre o '
                                                                   'Parlamentar.',
                            

,0
ListaParlamentarEmExercicio.noNamespaceSchemaLocation,https://legis.senado.leg.br/dadosabertos/dados...
ListaParlamentarEmExercicio.Metadados.Versao,09/10/2026 10:12:43
ListaParlamentarEmExercicio.Metadados.VersaoServico,4
ListaParlamentarEmExercicio.Metadados.DataVersaoServico,2020-07-15
ListaParlamentarEmExercicio.Metadados.DescricaoDataSet,Lista dos Parlamentares que estão atualmente e...
ListaParlamentarEmExercicio.Parlamentares.Parlamentar,[{'IdentificacaoParlamentar': {'CodigoParlamen...


## 3. Inspecionar recursos legislativos prioritários

Localize no OpenAPI rotas relacionadas a matérias, tramitações, votação, comissões, sessões e discursos. Não presuma os caminhos nem a equivalência semântica com a Câmara.

In [5]:
terms = ['materia', 'tramita', 'vota', 'comissao', 'sessao', 'discurso', 'parlamentar', 'senador']
mask = routes_df['path'].str.lower().apply(lambda path: any(term in path for term in terms))
display(routes_df[mask])

# Escolha uma rota GET sem parâmetros obrigatórios e faça uma chamada curta.
# Revise primeiro operation['parameters']; passe apenas filtros seguros/pequenos.
candidates = [r for r in rows if r['method'] == 'GET' and any(t in r['path'].lower() for t in ['materia', 'senador'])]
pprint(candidates[:20])


,tag,method,path,summary
0,Comissão,GET,/dadosabertos/comissao/agenda/atual/iCal,Obtém a Agenda das Comissões do Congresso Naci...
1,Comissão,GET,/dadosabertos/comissao/agenda/mes/{mesReferencia},Agenda das Reuniões de Comissão no mês informado
2,Comissão,GET,/dadosabertos/comissao/agenda/{dataInicio}/{da...,Agenda das Reuniões de Comissão
3,Comissão,GET,/dadosabertos/comissao/agenda/{dataReferencia},Agenda das Reuniões de Comissão
4,Comissão,GET,/dadosabertos/comissao/cpi/{comissao}/requerim...,Lista Requerimentos de CPI. Exclusivo para CPIs
...,...,...,...,...
152,Votação,GET,/dadosabertos/senador/{codigo}/votacoes,(DEPRECATED) Obtém as Votações Nominais de um ...
153,Votação,GET,/dadosabertos/votacao,Votos Nominais de Parlamentares em Processos L...
154,Votação,GET,/dadosabertos/votacaoComissao/comissao/{siglaC...,Votações de Matérias nas Comissões do Senado F...
155,Votação,GET,/dadosabertos/votacaoComissao/materia/{sigla}/...,Votações de Matérias nas Comissões por Identif...


[{'method': 'GET',
  'path': '/dadosabertos/votacaoComissao/materia/{sigla}/{numero}/{ano}',
  'summary': 'Votações de Matérias nas Comissões por Identificação da Matéria',
  'tag': 'Votação'},
 {'method': 'GET',
  'path': '/dadosabertos/senador/{codigo}',
  'summary': 'Obtém informações de um Senador',
  'tag': 'Parlamentar'},
 {'method': 'GET',
  'path': '/dadosabertos/senador/{codigo}/votacoes',
  'summary': '(DEPRECATED) Obtém as Votações Nominais de um Senador',
  'tag': 'Votação'},
 {'method': 'GET',
  'path': '/dadosabertos/senador/{codigo}/relatorias',
  'summary': '(DEPRECATED) Obtém as matérias de Relatoria de um Senador',
  'tag': 'Processo, Parlamentar'},
 {'method': 'GET',
  'path': '/dadosabertos/senador/{codigo}/profissao',
  'summary': 'Obtém as Profissões de um Senador',
  'tag': 'Parlamentar'},
 {'method': 'GET',
  'path': '/dadosabertos/senador/{codigo}/mandatos',
  'summary': 'Obtém os Mandatos Parlamentares de um Senador',
  'tag': 'Parlamentar'},
 {'method': 'GET'

## 4. Catálogo de arquivos e anotações

Use a página oficial do [catálogo legislativo](https://www12.senado.leg.br/dados-abertos/conjuntos?portal=Legislativo) para comparar API com arquivos ZIP/CSV/XML/JSON. Para cada conjunto prioritário, registre URL, formato, periodicidade, período coberto, tamanho e se é mais completo que a API.

Não baixar lotes grandes até fechar escopo e estratégia de atualização. Anote também quais IDs podem relacionar matéria originada na Câmara e matéria em tramitação no Senado.

## Anotações dos outputs — 2026-10-09

- A especificação OpenAPI 3.1 respondeu HTTP 200 e listou 157 operações GET; 42 estão marcadas como depreciadas. Respeitar limite publicado de 10 requisições/s e prever 429/503.
- A lista de senadores em exercício respondeu com objeto JSON aninhado `ListaParlamentarEmExercicio`, metadados e 81 entradas. Os campos incluem `CodigoParlamentar` e `Mandato.CodigoMandato`; os códigos vêm como texto no feed.
- Consultas de detalhe, mandatos e filiações do senador `5672` responderam HTTP 200. A lista contém quatro filiações para esse senador.
- A lista de processos sem filtro produziu cerca de 8,6 MB. Não a execute novamente sem filtros; consulte detalhes por `id` ou restrinja a lista.
- A consulta ampla de votações por parlamentar produziu 423 registros (~918 KB). Com `codigoParlamentar=5672` e datas 2023-02-01 a 2023-02-28, retornou uma votação: `idProcesso=8361684`, `codigoMateria=155773`, `codigoSessao=320695`, `codigoSessaoVotacao=6674`. O voto aninhado contém `codigoParlamentar=5672`; detalhe do processo confirma `id=8361684` e `codigoMateria=155773`.
- Caminhos na especificação incluem `/dadosabertos`; ao montá-los use o host como raiz, sem duplicar esse prefixo.
- Ver `docs/chaves_relacionamentos.md` para evidência e limites e `docs/catalogo_recursos.csv` para inventário de APIs e arquivos.
